# Sim-folds

For each similarity threshold, the sequences with no other sequence of the dataset at or above
that similarity, split in train, validation and test partitions:

- ArchiveII: 10 random partitions, each with 20% of the sequences for test (test partitions
  overlap between folds).
- bpRNA: 10 folds with mutually exclusive test partitions (10% of the sequences each).

In [ ]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

# Splits generation
...and distribution analysis

In [ ]:
DATASET = "archiveII" # or "bpRNA"
DATA_PATH = "../../data/"
SCHEME = {"archiveII": "bootstrap", "bpRNA": "kfold"}[DATASET]
SEQSIM = {"archiveII": "seqsim_f_all.h5", "bpRNA": "bprna_seqsim_full_ACGU.h5"}[DATASET] # see the README for the download links
OUT_PATH = f"{DATA_PATH}{DATASET}/sim_folds_split/"

In [ ]:
dataset = pd.read_csv(f"{DATA_PATH}{DATASET}/{DATASET}.csv", index_col="id")
dataset["len"] = dataset.sequence.str.len()
dataset = dataset[['sequence', 'base_pairs', 'len']]

seq_sim = pd.read_hdf(f"{DATA_PATH}{DATASET}/distances/{SEQSIM}") # IDscore

MAX_L = 512 # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
upto = list(dataset.loc[dataset.len < MAX_L].index) # <<< =<
dataset = dataset.loc[upto]

seq_sim = seq_sim.loc[upto]
seq_sim = seq_sim[upto]

In [ ]:
fam_order = ["grp1", "tmRNA", "tRNA", "5s", "srp", "telomerase", "RNaseP", "16s", "23s"]

def plot_families(fam, no_similar, max_sim):
    # plot the distribution of the families
    counts = fam.groupby("family").count()
    #counts = counts/max(counts)
    cc = pd.DataFrame(counts)
    xorder2 = fam_order #list(cc.index)
    for f in fam_order:
        if f not in cc.index:
            cc.loc[f] = 0
    cc = cc.rename(columns={"len": "count"})

    fig = sns.set(rc={'figure.figsize':(18,5)})
    sns.set(font_scale=1.3)
    sns.set_style(style='white') 
    fig = plt.figure(figsize=(18,5))
    bp = sns.barplot(data=cc, y="count", x=cc.index, order=xorder2,
                    facecolor=(0.0,0.0,1.0,0.1), linewidth=1, 
                    edgecolor=(0.0,0.0,1.0,0.3), zorder=-1)
    bp.set_title(f"Similarity < {max_sim:.2} - {len(no_similar)} sequences - length: {fam.len.min():.0f} < {fam.len.mean():.0f} < {fam.len.max():.0f}")


In [ ]:
# set the diagonal to 0, to exclude the similarity of a sequence with itself
values = seq_sim.to_numpy(copy=True)
np.fill_diagonal(values, 0)
seq_sim = pd.DataFrame(values, index=seq_sim.index, columns=seq_sim.columns)

os.makedirs(OUT_PATH, exist_ok=True)

thresholds = np.linspace(0, 1, 11)[1:-1]
thresholds = thresholds[thresholds >= 0.4] # sim30 and below are not used
# for each threshold, extract the rows for which similarities in all columns are below that threshold
for max_sim in thresholds:
    max_sim_int = int(max_sim*100)
    no_similar = list(seq_sim[(seq_sim < max_sim).all(axis=1)].index)

    fam = pd.DataFrame(no_similar, columns=["id"])
    fam = fam.set_index("id")
    if len(fam) > 0:
        # extract the family name from the id
        fam.insert(loc=0, column='family', value=[id.split("_")[0] for id in fam.index])
        fam.insert(loc=1, column="len", value=dataset.len)

        # plot the distribution of the families
        if DATASET == "archiveII":
            plot_families(fam, no_similar, max_sim)

        # generate partitions for train, validation and test with sklearn
        folds = []
        if SCHEME == "bootstrap":
            for k in range(10):
                train, test = train_test_split(fam, test_size=0.2, random_state=3003+k)
                train, valid = train_test_split(train, test_size=0.2, random_state=3003+k)
                folds.append((train, valid, test))
        else:
            # 10 folds with mutually exclusive test partitions
            skf = StratifiedKFold(n_splits=min(len(fam), 10), shuffle=True, random_state=3003)
            for train_index, test_index in skf.split(fam, np.ones((len(fam),))):
                train, test = fam.iloc[train_index], fam.iloc[test_index]
                train, valid = train_test_split(train, test_size=0.1, random_state=3003)
                folds.append((train, valid, test))

        split_max_sim = pd.concat([pd.DataFrame({"fold_name": k, "fold_number": k, "partition": partition, "id": part.index})
                                   for k, (train, valid, test) in enumerate(folds)
                                   for partition, part in [("train", train), ("valid", valid), ("test", test)]])
        split_max_sim.to_csv(f"{OUT_PATH}sim{max_sim_int}.csv", index=False)